## Código 3 — Diagnóstico del residuo por fracción difusa

Este código analiza el error del modelo NOCT usando la base generada en el Código 2.

Primero filtra las horas válidas para modelado. Luego calcula métricas globales y por régimen de fracción difusa (baja fracción difusa, régimen mixto y alta fracción difusa), incluyendo MAE, RMSE, bias, R² y error relativo promedio.

Las métricas absolutas usan todas las filas físicamente válidas. El error relativo promedio se calcula solo en las filas donde `relative_error` está definido, es decir, donde `p_ref_dc > 50 W`.

También calcula la correlación entre `fd` y el error del modelo, y genera gráficos para visualizar:

- residuo del modelo frente a la fracción difusa;
- error absoluto por régimen;
- error relativo por régimen;
- comparación entre la potencia estimada por NOCT y la referencia PVWatts.

El objetivo es evaluar si el error del modelo físico cambia según la fracción difusa antes de entrenar el modelo híbrido.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


INPUT_FILE = Path("data/02_noct_vs_pvwatts_santiago.csv")

OUT_DIR = Path("outputs/03_diagnostico_fd_santiago")
OUT_DIR.mkdir(parents=True, exist_ok=True)

METRICS_GLOBAL_FILE = OUT_DIR / "metricas_globales_santiago.csv"
METRICS_REGIMEN_FILE = OUT_DIR / "metricas_por_regimen_fd_santiago.csv"
CORRELATIONS_FILE = OUT_DIR / "correlaciones_fd_error_santiago.csv"

REGIMEN_ORDER = ["baja_fd", "mixto", "alta_fd"]
REGIMEN_LABELS = {
    "baja_fd": "Baja fracción difusa",
    "mixto": "Régimen mixto",
    "alta_fd": "Alta fracción difusa",
}



# FUNCIONES DE METRICAS
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def bias(y_true, y_pred):
    # positivo = NOCT sobreestima si usamos pred - true
    return np.mean(y_pred - y_true)


def r2_score_manual(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)

    if ss_tot == 0:
        return np.nan

    return 1 - ss_res / ss_tot


def calculate_metrics(df):
    y_true = df["p_ref_dc"]
    y_pred = df["p_noct"]

    return {
        "n": len(df),
        "n_error_relativo": df["relative_error"].notna().sum(),
        "MAE_W": mae(y_true, y_pred),
        "RMSE_W": rmse(y_true, y_pred),
        "Bias_W": bias(y_true, y_pred),
        "R2": r2_score_manual(y_true, y_pred),
        "Error_relativo_promedio": df["relative_error"].mean(),
        "P_ref_promedio_W": y_true.mean(),
        "P_NOCT_promedio_W": y_pred.mean(),
        "fd_promedio": df["fd"].mean(),
    }



# CARGA Y FILTRO
def load_data():
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"No se encontró el archivo {INPUT_FILE}. "
            "Primero ejecuta el Código 2 para Santiago."
        )

    df = pd.read_csv(INPUT_FILE)

    required_columns = [
        "fd",
        "regimen_fd",
        "p_ref_dc",
        "p_noct",
        "epsilon",
        "abs_epsilon",
        "relative_error",
        "is_valid_model",
    ]

    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(f"Faltan columnas requeridas: {missing}")

    # Por si viene como texto
    if df["is_valid_model"].dtype == "object":
        df["is_valid_model"] = df["is_valid_model"].astype(str).str.lower() == "true"

    df["regimen_fd"] = pd.Categorical(
        df["regimen_fd"],
        categories=REGIMEN_ORDER,
        ordered=True,
    )

    df_valid = df[df["is_valid_model"]].copy()

    # Las metricas absolutas usan todas las filas fisicamente validas.
    # El error relativo se calcula solo donde relative_error no es NaN.
    df_valid = df_valid.dropna(
        subset=[
            "fd",
            "regimen_fd",
            "p_ref_dc",
            "p_noct",
            "epsilon",
            "abs_epsilon",
        ]
    )

    return df, df_valid



# TABLAS
def build_metrics_tables(df_valid):
    metrics_global = pd.DataFrame([calculate_metrics(df_valid)])
    metrics_global.to_csv(METRICS_GLOBAL_FILE, index=False)

    rows = []

    for regimen in REGIMEN_ORDER:
        group = df_valid[df_valid["regimen_fd"] == regimen].copy()

        if len(group) == 0:
            continue

        row = calculate_metrics(group)
        row["regimen_fd"] = regimen
        rows.append(row)

    metrics_regimen = pd.DataFrame(rows)

    cols = ["regimen_fd"] + [c for c in metrics_regimen.columns if c != "regimen_fd"]
    metrics_regimen = metrics_regimen[cols]

    metrics_regimen.to_csv(METRICS_REGIMEN_FILE, index=False)

    return metrics_global, metrics_regimen


def build_correlations(df_valid):
    correlations = []

    variables_error = ["epsilon", "abs_epsilon", "relative_error"]

    for var in variables_error:
        pearson = df_valid["fd"].corr(df_valid[var], method="pearson")
        spearman = df_valid["fd"].corr(df_valid[var], method="spearman")

        correlations.append({
            "variable_error": var,
            "corr_pearson_con_fd": pearson,
            "corr_spearman_con_fd": spearman,
        })

    corr_df = pd.DataFrame(correlations)
    corr_df.to_csv(CORRELATIONS_FILE, index=False)

    return corr_df



# GRAFICOS
def plot_epsilon_vs_fd(df_valid):
    plt.figure(figsize=(8, 5))
    plt.scatter(df_valid["fd"], df_valid["epsilon"], s=8, alpha=0.35)
    plt.axhline(0, linestyle="--", linewidth=1)

    plt.xlabel("Fracción difusa (DHI/GHI)")
    plt.ylabel("Residuo: referencia PVWatts - estimación NOCT [W]")
    plt.title("Santiago: residuo del modelo físico NOCT según fracción difusa")
    plt.tight_layout()

    plt.savefig(OUT_DIR / "scatter_epsilon_vs_fd_santiago.png", dpi=300)
    plt.close()


def plot_abs_error_by_regimen(df_valid):
    data = [
        df_valid.loc[df_valid["regimen_fd"] == reg, "abs_epsilon"]
        for reg in REGIMEN_ORDER
    ]

    plt.figure(figsize=(8, 5))
    plt.boxplot(data, tick_labels=[REGIMEN_LABELS[reg] for reg in REGIMEN_ORDER], showfliers=False)

    plt.xlabel("Régimen de fracción difusa")
    plt.ylabel("Error absoluto del modelo NOCT [W]")
    plt.title("Santiago: distribución del error absoluto por régimen radiativo")
    plt.tight_layout()

    plt.savefig(OUT_DIR / "boxplot_abs_error_por_regimen_santiago.png", dpi=300)
    plt.close()


def plot_relative_error_by_regimen(df_valid):
    data = [
        df_valid.loc[df_valid["regimen_fd"] == reg, "relative_error"].dropna() * 100
        for reg in REGIMEN_ORDER
    ]

    plt.figure(figsize=(8, 5))
    plt.boxplot(data, tick_labels=[REGIMEN_LABELS[reg] for reg in REGIMEN_ORDER], showfliers=False)

    plt.xlabel("Régimen de fracción difusa")
    plt.ylabel("Error relativo [%]")
    plt.title("Santiago: distribución del error relativo por régimen radiativo")
    plt.tight_layout()

    plt.savefig(OUT_DIR / "boxplot_error_relativo_por_regimen_santiago.png", dpi=300)
    plt.close()


def plot_pred_vs_ref(df_valid):
    plt.figure(figsize=(6, 6))
    plt.scatter(df_valid["p_ref_dc"], df_valid["p_noct"], s=8, alpha=0.35)

    max_val = max(df_valid["p_ref_dc"].max(), df_valid["p_noct"].max())
    plt.plot([0, max_val], [0, max_val], linestyle="--", linewidth=1)

    plt.xlabel("Potencia de referencia PVWatts DC [W]")
    plt.ylabel("Potencia estimada por NOCT [W]")
    plt.title("Santiago: modelo físico NOCT frente a referencia PVWatts")
    plt.tight_layout()

    plt.savefig(OUT_DIR / "scatter_pnoct_vs_pref_santiago.png", dpi=300)
    plt.close()


def plot_mae_by_regimen(metrics_regimen):
    metrics_regimen = metrics_regimen.copy()
    metrics_regimen["regimen_fd"] = pd.Categorical(
        metrics_regimen["regimen_fd"],
        categories=REGIMEN_ORDER,
        ordered=True,
    )
    metrics_regimen = metrics_regimen.sort_values("regimen_fd")

    plt.figure(figsize=(8, 5))
    plt.bar(metrics_regimen["regimen_fd"].map(REGIMEN_LABELS), metrics_regimen["MAE_W"])

    plt.xlabel("Régimen de fracción difusa")
    plt.ylabel("Error absoluto medio (MAE) [W]")
    plt.title("Santiago: MAE del modelo físico NOCT por régimen radiativo")
    plt.tight_layout()

    plt.savefig(OUT_DIR / "bar_mae_por_regimen_santiago.png", dpi=300)
    plt.close()



# EJECUCION
def main():
    print("Leyendo datos...")
    df, df_valid = load_data()

    print("\nFilas totales:")
    print(len(df))

    print("\nFilas válidas para diagnóstico:")
    print(len(df_valid))

    print("\nFilas con error relativo definido:")
    print(df_valid["relative_error"].notna().sum())

    print("\nConteo por régimen fd:")
    print(df_valid["regimen_fd"].value_counts().sort_index())

    print("\nCalculando métricas...")
    metrics_global, metrics_regimen = build_metrics_tables(df_valid)

    print("\nMétricas globales:")
    print(metrics_global.round(4).to_string(index=False))

    print("\nMétricas por régimen fd:")
    print(metrics_regimen.round(4).to_string(index=False))

    print("\nCalculando correlaciones...")
    corr_df = build_correlations(df_valid)

    print("\nCorrelaciones fd vs error:")
    print(corr_df.round(4).to_string(index=False))

    print("\nGenerando gráficos...")
    plot_epsilon_vs_fd(df_valid)
    plot_abs_error_by_regimen(df_valid)
    plot_relative_error_by_regimen(df_valid)
    plot_pred_vs_ref(df_valid)
    plot_mae_by_regimen(metrics_regimen)

    print("\nArchivos generados en:")
    print(OUT_DIR)

    print("\nTablas:")
    print(METRICS_GLOBAL_FILE)
    print(METRICS_REGIMEN_FILE)
    print(CORRELATIONS_FILE)

    print("\nGráficos:")
    for file in OUT_DIR.glob("*.png"):
        print(file)


if __name__ == "__main__":
    main()

Leyendo datos...

Filas totales:
8760

Filas válidas para diagnóstico:
4215

Filas con error relativo definido:
3688

Conteo por régimen fd:
regimen_fd
baja_fd    1349
mixto      1126
alta_fd    1740
Name: count, dtype: int64

Calculando métricas...



Métricas globales:
   n  n_error_relativo   MAE_W  RMSE_W  Bias_W     R2  Error_relativo_promedio  P_ref_promedio_W  P_NOCT_promedio_W  fd_promedio
4215              3688 11.0507 15.0507 -1.6049 0.9968                   0.0372          365.7887           364.1838       0.5797

Métricas por régimen fd:
regimen_fd    n  n_error_relativo   MAE_W  RMSE_W   Bias_W     R2  Error_relativo_promedio  P_ref_promedio_W  P_NOCT_promedio_W  fd_promedio
   baja_fd 1349              1349 20.4243 23.9702 -14.5404 0.9589                   0.0298          673.9032           659.3628       0.1960
     mixto 1126              1124  7.8833  9.6674   2.5096 0.9967                   0.0307          361.5111           364.0207       0.4873
   alta_fd 1740              1215  5.8331  6.5417   5.7614 0.9967                   0.0515          129.6795           135.4409       0.9370

Calculando correlaciones...



Correlaciones fd vs error:
variable_error  corr_pearson_con_fd  corr_spearman_con_fd
       epsilon              -0.5610               -0.5015
   abs_epsilon              -0.5776               -0.5729
relative_error               0.3590                0.3119

Generando gráficos...



Archivos generados en:
outputs\03_diagnostico_fd_santiago

Tablas:
outputs\03_diagnostico_fd_santiago\metricas_globales_santiago.csv
outputs\03_diagnostico_fd_santiago\metricas_por_regimen_fd_santiago.csv
outputs\03_diagnostico_fd_santiago\correlaciones_fd_error_santiago.csv

Gráficos:
outputs\03_diagnostico_fd_santiago\bar_mae_por_regimen_santiago.png
outputs\03_diagnostico_fd_santiago\boxplot_abs_error_por_regimen_santiago.png
outputs\03_diagnostico_fd_santiago\boxplot_error_relativo_por_regimen_santiago.png
outputs\03_diagnostico_fd_santiago\scatter_epsilon_vs_fd_santiago.png
outputs\03_diagnostico_fd_santiago\scatter_pnoct_vs_pref_santiago.png
